In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
text = "Tokenization splits text into pieces a model can understand."
encoded = tokenizer(text)
print(encoded)

{'input_ids': [101, 19204, 3989, 19584, 3793, 2046, 4109, 1037, 2944, 2064, 3305, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [10]:
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
print(tokens)

['[CLS]', 'token', '##ization', 'splits', 'text', 'into', 'pieces', 'a', 'model', 'can', 'understand', '.', '[SEP]']


In [11]:
print("Tokens:        ", tokens)
print("Attention mask:", encoded["attention_mask"])
print("Token type ids:", encoded["token_type_ids"])

Tokens:         ['[CLS]', 'token', '##ization', 'splits', 'text', 'into', 'pieces', 'a', 'model', 'can', 'understand', '.', '[SEP]']
Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Token type ids: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [12]:
batch = ["Short one.", "This sentence is quite a bit longer than the first one."]
batch_encoded = tokenizer(batch, padding=True, return_tensors="pt")

print(batch_encoded["input_ids"])
print(batch_encoded["attention_mask"])

tensor([[ 101, 2460, 2028, 1012,  102,    0,    0,    0,    0,    0,    0,    0,
            0,    0],
        [ 101, 2023, 6251, 2003, 3243, 1037, 2978, 2936, 2084, 1996, 2034, 2028,
         1012,  102]])
tensor([[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [13]:
from transformers import AutoModel

model = AutoModel.from_pretrained("bert-base-uncased")
model.eval()

with torch.no_grad():
    output = model(**batch_encoded)

print(output.last_hidden_state.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([2, 14, 768])


In [14]:
# Method 1: CLS pooling — just take the first token's vector
cls_embedding = output.last_hidden_state[:, 0, :]
print("CLS embedding shape:", cls_embedding.shape)

# Method 2: Mean pooling — average all real (non-padding) token vectors
mask = batch_encoded["attention_mask"].unsqueeze(-1)
summed = (output.last_hidden_state * mask).sum(dim=1)
counts = mask.sum(dim=1)
mean_pooled = summed / counts
print("Mean-pooled embedding shape:", mean_pooled.shape)

CLS embedding shape: torch.Size([2, 768])
Mean-pooled embedding shape: torch.Size([2, 768])
